# MODELOS.

In [18]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV 
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score


In [19]:
df = pd.read_csv("../data/coches_proc_1.csv")
df.head()

,make,model,version,price,fuel,year,kms,power,doors,shift,color,is_professional
0,CITROEN,C1,CITROEN C1 PureTech 60KW 82CV Feel 5p.,6200,Gasolina,2017.0,50071,82.0,5,Manual,Blanco,True
1,FORD,Transit Connect,FORD Transit Connect Van 1.5 TDCi 100cv Ambien...,7851,Diésel,2016.0,103000,100.0,4,Manual,Blanco,True
2,VOLKSWAGEN,Caravelle,VOLKSWAGEN Caravelle Largo 2.0 TDI 140 Comfort...,19426,Diésel,2014.0,120000,140.0,4,Manual,Blanco,True
3,FORD,Transit,FORD Transit 350 96kW L4 Ambiente Propulsion T...,22850,Diésel,2017.0,107000,130.0,2,Manual,Blanco,True
4,PEUGEOT,3008,PEUGEOT 3008 Style 1.2 PureTech 130 SS 5p.,11490,Gasolina,2016.0,78665,130.0,5,Manual,Blanco,True


___________________________________________________________________________________

## MODELO DE REGRESIÓN LINEAL

In [20]:
X = df[["year", "kms", "power", "is_professional"]]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(33000, 4)
(33000,)
(8251, 4)
(8251,)


In [21]:
modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)
pred_lr = modelo_lr.predict(X_test)


print("MAE: ", mean_absolute_error(y_test, pred_lr))
print("MSE: ", mean_squared_error(y_test, pred_lr))
print("MAPE: ", mean_absolute_percentage_error(y_test, pred_lr))
print("R2: ", r2_score(y_test, pred_lr))

MAE:  5049.873480498633
MSE:  119466736.2541861
MAPE:  0.5818407351879382
R2:  0.6360719998315894


### GUARDAMOS EL MODELO.

In [22]:
with open("../modelos/modelo_rl.pkl", "wb") as f:
    pickle.dump(modelo_lr, f)

__________________________________________________________

## Modelo regresión polinomica.

In [23]:
modelo_rp = PolynomialFeatures(degree=3)
modelo_rp.fit(X_train)
X_train_rp = modelo_rp.transform(X_train)
X_test_rp = modelo_rp.transform(X_test)

modelo_rp_lr = LinearRegression()
modelo_rp_lr.fit(X_train_rp, y_train)
pred_rp = modelo_rp_lr.predict(X_test_rp)

print("MAE: ", mean_absolute_error(y_test, pred_rp))
print("MSE: ", mean_squared_error(y_test, pred_rp))
print("MAPE: ", mean_absolute_percentage_error(y_test, pred_rp))
print("R2: ", r2_score(y_test, pred_rp))

MAE:  4023.8534030881924
MSE:  68528670.91517323
MAPE:  0.3529537562441457
R2:  0.7912431280679249


### GUARDAMOS EL MODELO.

In [24]:
with open("../modelos/modelo_rp.pkl", "wb") as f:
    pickle.dump(modelo_rp_lr, f)

______________________________________________________________________________

## PIPELINE.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures()),
    ("model", LinearRegression())
])

params = [
    {
        "scaler":[None, StandardScaler()],
        "poly__degree":[1,2,3,4,5],
        "model": [LinearRegression()]
    },
    {
        "scaler":[None, StandardScaler()],
        "poly__degree":[1,2,3,4,5],
        "model": [Ridge()],
        "model__alpha":[0.1,1,10]
    },
    {
        "scaler":[None, StandardScaler()],
        "poly__degree":[1,2,3,4,5],
        "model": [Lasso()],
        "model__alpha":[0.1,1,10]
    }
]

grid_search = GridSearchCV(pipeline, params, scoring="neg_mean_absolute_error", n_jobs=1, cv = 5, verbose= 1)

grid_search.fit(X_train, y_train)

print("Mejores parametros: ", grid_search.best_params_)
print("Mejores valores: ", grid_search.best_score_)




Fitting 5 folds for each of 70 candidates, totalling 350 fits


c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=1.19658e-27): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=1.22332e-27): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=1.20806e-27): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=1.20013e-27): result may not be accurat

In [ ]:
modelo_pipeline = grid_search.best_params_

In [ ]:
with open("../modelos/modelo_pipeline.pkl", "wb") as f:
    pickle.dump(modelo_pipeline, f)